Data Split

In [1]:
import pandas as pd
import os
import numpy as np

def load_data_fixed(image_folder, csv_path):
    print(f"Chargement de {csv_path}...")
    
    # 1. Lire le CSV
    df = pd.read_csv(csv_path)
    
    # 2. Configuration des NOMS DE COLONNES (Ceux que l'erreur nous a donnés)
    col_id = 'frame_id'  
    col_x = 'x'
    col_y = 'y'
    
    # Sécurité : Vérifie que les colonnes sont bien là
    if col_id not in df.columns:
        print(f"ERREUR : Colonne '{col_id}' toujours introuvable.")
        return [], []

    # 3. Création des chemins d'images
    # IMPORTANT : Je suppose que tes images sont des .jpg
    # Si ce sont des .png, change '.jpg' en '.png' ci-dessous
    image_paths = []
    for name in df[col_id]:
        # On construit : Dossier + frame_id + .jpg
        # Ex: "C:\Data\...\1050.jpg"
        full_path = os.path.join(image_folder, str(name) + ".jpg")
        image_paths.append(full_path)
    
    # 4. Récupération des labels x, y
    labels = df[[col_x, col_y]].values.astype('float32')
    
    return image_paths, labels.tolist()

# --- UTILISATION ---

path_img_p2 = r"C:\Users\Yassin\Desktop\eye tracking\data\yass\yassine"
path_csv_p2 = r"C:\Users\Yassin\Desktop\eye tracking\data\yass\Excel_yassine\SCtes_t_.csv"

path_img_p3 = r"C:\Users\Yassin\Desktop\eye tracking\data\Stev22\Stev2"
path_csv_p3 = r"C:\Users\Yassin\Desktop\eye tracking\data\Stev22\SC2_.csv"

# 1. Charger Personne 2
images_p2, labels_p2 = load_data_fixed(path_img_p2, path_csv_p2)
print(f"--> Personne 2 : {len(images_p2)} images trouvées")

# 2. Charger Personne 3
images_p3, labels_p3 = load_data_fixed(path_img_p3, path_csv_p3)
print(f"--> Personne 3 : {len(images_p3)} images trouvées")

# 3. Fusionner pour le Domaine B
if len(images_p2) > 0 and len(images_p3) > 0:
    all_images_B = images_p2 + images_p3
    all_labels_B = np.array(labels_p2 + labels_p3, dtype=np.float32)
    print("="*30)
    print(f"SUCCÈS : TOTAL DOMAINE B = {len(all_images_B)} images.")
    print("="*30)
else:
    print("ATTENTION : Une des listes est vide. Vérifie les chemins.")

Chargement de C:\Users\Yassin\Desktop\eye tracking\data\yass\Excel_yassine\SCtes_t_.csv...
--> Personne 2 : 3567 images trouvées
Chargement de C:\Users\Yassin\Desktop\eye tracking\data\Stev22\SC2_.csv...
--> Personne 3 : 7149 images trouvées
SUCCÈS : TOTAL DOMAINE B = 10716 images.


Calculer la "Baseline" (Budget 0%)

In [2]:
pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [4]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from tqdm import tqdm # Barre de chargement

# ==========================================
# 1. LA DIVISION (SPLIT)
# ==========================================
print("Division des données en cours...")

# A. On isole 20% pour le TEST FINAL (Ne jamais toucher !)
X_remaining, X_test_B, y_remaining, y_test_B = train_test_split(
    all_images_B, all_labels_B, test_size=0.20, random_state=42
)

# B. On divise le reste en POOL (70% global) et VALIDATION (10% global)
# 0.125 * 0.80 = 0.10 (10% du total)
X_pool, X_val_B, y_pool, y_val_B = train_test_split(
    X_remaining, y_remaining, test_size=0.125, random_state=42
)

print(f"--> Test Set (Pour la note finale) : {len(X_test_B)} images")
print(f"--> Validation Set (Pour le suivi) : {len(X_val_B)} images")
print(f"--> Active Learning Pool (Réservoir) : {len(X_pool)} images")

# ==========================================
# 2. PRÉPARATION DU TEST
# ==========================================

test_dataset_B = GazeDataset(X_test_B, y_test_B, transform=data_transforms['val']) 
test_loader_B = DataLoader(test_dataset_B, batch_size=32, shuffle=False)

# ==========================================
# 3. CALCUL DE LA BASELINE
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GazeModel() # Ton modèle
model.load_state_dict(torch.load("gaze_model.pth", map_location=DEVICE))
model.to(DEVICE)
model.eval() # Mode évaluation

criterion = nn.MSELoss()
total_loss = 0.0
total_samples = 0

print("\nCalcul de la Baseline en cours...")
with torch.no_grad():
    # La barre de chargement (tqdm) va s'afficher ici
    for images, labels in tqdm(test_loader_B):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)
        
        total_loss += loss.item() * images.size(0)
        total_samples += images.size(0)

baseline_mse = total_loss / total_samples

print("\n" + "="*40)
print(f"RÉSULTAT BASELINE (Point de départ) :")
print(f"MSE Loss : {baseline_mse:.4f}")
print("="*40)

Division des données en cours...
--> Test Set (Pour la note finale) : 2144 images
--> Validation Set (Pour le suivi) : 1072 images
--> Active Learning Pool (Réservoir) : 7500 images


NameError: name 'GazeDataset' is not defined